# 🚀 AI Resume Analyzer and Job Recommendation System using NLP

## 📌 Project Overview
This notebook demonstrates the underlying NLP pipelines, keyword parsing, skill extraction, and similarity matching engines used in the **AI Resume Analyzer & Job Recommendation System**. 

### 🎓 Course/College Submission Details:
- **Project Title**: AI Resume Analyzer and Job Recommendation System using NLP
- **Developer**: Skandhan M U
- **Academic Year**: 2026
- **Course**: B.Tech Computer Science & Engineering

---

## 🛠️ Step 1: Install Dependencies
We need standard NLP and document parsing libraries including `spaCy`, `pypdf`, `python-docx`, `scikit-learn`, `reportlab`, and `python-pptx`.

In [ ]:
!pip install pypdf python-docx spacy scikit-learn reportlab python-pptx matplotlib plotly pandas numpy python-dotenv
!python -m spacy download en_core_web_sm

## 📚 Step 2: Import Core Libraries

In [ ]:
import re
import pandas as pd
import numpy as np
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader
from docx import Document
import matplotlib.pyplot as plt

print("All libraries imported successfully!")

## 🧹 Step 3: Text Cleaning and Preprocessing Module
Removes emails, phone numbers, URLs, and performs tokenization/lemmatization using spaCy.

In [ ]:
nlp = spacy.load("en_core_web_sm")

def clean_text(text):
    text = text.lower()
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Remove Emails
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' ', text)
    # Remove Phones
    text = re.sub(r'\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b', ' ', text)
    # Remove special characters
    text = re.sub(r'[^\w\s\-\.\,\/\+]', ' ', text)
    return text.strip()

def preprocess_text(text):
    cleaned = clean_text(text)
    doc = nlp(cleaned)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.text.strip()]
    return " ".join(tokens)

test_cv = "Contact: user@domain.com | Tel: +1 555-019-2834. Visit website http://github.com/myprofile. Skilled in Python and Deep Learning."
print("Raw Text: ", test_cv)
print("Processed Text:", preprocess_text(test_cv))

## 🏷️ Step 4: Section Segmentation
Divides resume contents into Education, Experience, Projects, and Certifications.

In [ ]:
def extract_sections(text):
    sections = {"Education": "", "Experience": "", "Projects": "", "Certifications": "", "Contact Info": ""}
    patterns = {
        "Education": r'\b(?:education|academic background|degrees)\b',
        "Experience": r'\b(?:experience|work experience|employment history)\b',
        "Projects": r'\b(?:projects|academic projects|personal projects)\b',
        "Certifications": r'\b(?:certifications|licenses|credentials|courses)\b'
    }
    
    lines = text.split('\n')
    current_section = "Contact Info"
    section_lines = {key: [] for key in sections.keys()}
    
    for line in lines:
        stripped = line.strip()
        if not stripped: continue
        
        matched = False
        for sec_name, pattern in patterns.items():
            if re.match(pattern, stripped, re.IGNORECASE) and len(stripped) < 40:
                current_section = sec_name
                matched = True
                break
        if not matched:
            section_lines[current_section].append(line)
            
    for key in sections.keys():
        sections[key] = "\n".join(section_lines[key]).strip()
    return sections

sample_resume_content = """
Skandhan M U
Email: skandhan@example.com
Experience
Software Engineer at Google
Developed deep learning models and python web scrapers.
Education
B.Tech in Computer Science
"""
print(extract_sections(sample_resume_content))

## 🧬 Step 5: Dictionary-Based Skill Extraction
Using a dictionary of skills to match terms precisely while ignoring substring collisions.

In [ ]:
# Minimal skill dictionary mapping for demonstration
demo_skill_dictionary = {
    "python": "Programming",
    "deep learning": "AI",
    "sql": "Database",
    "pytorch": "DL",
    "scikit-learn": "Libraries",
    "docker": "DevOps"
}

def extract_skills_demo(text, skill_dict):
    text_lower = text.lower()
    detected = set()
    
    for skill in sorted(skill_dict.keys(), key=len, reverse=True):
        # Construct clean regex boundaries
        pattern = r'\b' + re.escape(skill) + r'\b'
        if re.search(pattern, text_lower):
            detected.add(skill)
            
    return sorted(list(detected))

print("Extracted Skills:", extract_skills_demo(test_cv, demo_skill_dictionary))

## 🎯 Step 6: TF-IDF & Cosine Similarity Match Engine
This section outlines matching logic combining NLP vectors and explicit skill overlap percentages.

In [ ]:
# Mock job role specifications
jobs = [
    {"Role": "Machine Learning Engineer", "Required": ["python", "pytorch", "deep learning", "scikit-learn"], "Description": "developing neural networks and machine learning applications in pytorch"},
    {"Role": "Software Engineer", "Required": ["python", "sql", "docker"], "Description": "software developer working on databases sql backend systems and Docker tools"}
]

def match_resume(resume_text, extracted_skills, job_roles):
    results = []
    cleaned_resume = preprocess_text(resume_text)
    
    for job in job_roles:
        role_desc = preprocess_text(job['Role'] + " " + " ".join(job['Required']) + " " + job['Description'])
        
        # TF-IDF Cosine Similarity
        vectorizer = TfidfVectorizer()
        tfidf = vectorizer.fit_transform([cleaned_resume, role_desc])
        cos_score = cosine_similarity(tfidf[0:1], tfidf[1:2]).flatten()[0]
        
        # Skill overlap
        matched = [s for s in job['Required'] if s in extracted_skills]
        missing = [s for s in job['Required'] if s not in extracted_skills]
        overlap_score = len(matched) / len(job['Required']) if job['Required'] else 0
        
        # Combined match score (40% TF-IDF, 60% Skills Overlap)
        final_score = (0.4 * cos_score + 0.6 * overlap_score) * 100
        
        results.append({
            "Role": job['Role'],
            "Match Score": round(final_score, 2),
            "Matched Skills": matched,
            "Missing Skills": missing
        })
    return sorted(results, key=lambda x: x['Match Score'], reverse=True)

candidate_cv = "Talented engineer skilled in Python development. Proficient in deep learning applications, neural networks, and PyTorch frameworks."
candidate_skills = extract_skills_demo(candidate_cv, demo_skill_dictionary)
print("Job Rankings for Candidate:\n", match_resume(candidate_cv, candidate_skills, jobs))

## 📈 Step 7: How to Run the Streamlit Dashboard Locally
1. Save the codebase structure as defined in the main project files.
2. Execute the setup command in your terminal:
   ```powershell
   pip install -r requirements.txt
   streamlit run app.py
   ```
3. Open the locally hosted web address in your browser.

## 🎓 Conclusion
This hybrid system matches candidates accurately, eliminates resume keyword spamming, and recommends explicit, actionable study guides to bridge career gaps.